# Interactive Denoising Lab — PandaOmron + GRPO LoRA — seed 103067, full_pool4_v5 `iter_0012` (v5)

Seed-sweep analysis of one GRPO LoRA checkpoint **plus the base model (no LoRA)**,
against one shared observation: the **first observation of GRPO scene seed 103067**,
rebuilt from the seed (no saved `.npz` needed). This is the v4 notebook's seed-sweep
section (its solver-convergence and residual-roughness sections are dropped), pointed at

`grpo_data/full_pool4_v5/lr5e-5PAWS_…_maskloss_fixedstd_iternorm_res7_velanch_1.0_i3/iter_0012`

CoffeeServeMug, 4-scene pool from seed 100067, vel-anchor `c=1.0` on
`full_pool4_v3/…_failfilter/iter_0003`, resumed from `…_iternorm/iter_0007`.
Seed 103067 is pool slot 3 (`100067 + 3·1000`), and the run had fast-forward off
(`fast_forward_pct 0.0`), so this observation is exactly where every 103067
training rollout started.

Each run (base and checkpoint) prints the per-component seed-spread stats and
**jerk / smoothness stats** in the same format, then draws **one composite figure**
(large 3D EEF-trajectory panel + per-component traces + mean±std spread).

## Where the observation comes from (Cell 1.5)

robocasa isn't importable in this kernel's venv, so Cell 1.5 runs
`scripts/denoising_lab/eval/capture_seed_obs.py` in the robocasa venv as a
subprocess. It builds the scene with the GRPO collector's own env factory and
reset sequence (`clear_ep_meta` → `reset(seed)` → `get_scene_bundle` →
`apply_scene_bundle`), then writes the observation to `OBS_DIR` in the
`interactive_rollout.py` npz format. Reruns reuse that file; pass `force=True`
to rebuild it. The printed `scene:` fingerprint should match the training log's
`scene:` line for this seed; set `EXPECTED_SCENE` to have the cell check it.

## How this works — one model load, one feature encode

The Eagle VLM backbone is **frozen and never touched by LoRA** — LoRA only wraps
`nn.Linear` modules inside the DiT action head. That makes a single-notebook sweep
straightforward:

1. **Load the 3B model once** (Cell 2).
2. **Inject LoRA once** at `rank=16, alpha=32` and **do NOT merge** (Cell 2.5).
   Merging folds the adapter into the base weights *irreversibly*; we keep it as an
   overlay so each checkpoint's weights can be swapped in for its run — and so the
   adapter can be *disabled* to recover the base model.
3. **Encode backbone features once per observation** (inside `analyze_all_checkpoints`).
   Because LoRA never touches the backbone, the same `features` are valid input for
   every checkpoint and for the base model.
4. **Base model (no LoRA)**: `analyze_all_checkpoints` first runs a sweep with the
   injected adapters *disabled* (via `disabled_adapters`), so the DiT produces its
   pretrained velocity field. This needs no second model copy and is independent of
   whichever checkpoint happens to be loaded.
5. **For each checkpoint**: `load_lora_checkpoint(...)` overwrites the adapter's A/B
   matrices in place (fully replacing the previous checkpoint), then we run the
   100-seed sweep and draw **one composite figure per checkpoint**.

`load_lora_checkpoint` does a strict two-sided key + shape check, so a checkpoint
trained with a different rank/target-module set would hard-fail here rather than
silently degrade. This run's TensorBoard config table records `lora_rank 16`,
`lora_alpha 32`, `lora_dropout 0.0` and the default target modules, matching Cell 1.

> ⚠️ The LoRA **scale** (`alpha / rank`) comes from the `apply_lora_to_dit` call,
> *not* from the checkpoint file. If you add a checkpoint trained with a different
> `alpha`, set `LORA_ALPHA` to match it before loading — otherwise the adapter is
> applied at the wrong scale.

To analyze another pool scene, call
`analyze_all_checkpoints(capture_seed_observation(100067))` in a new cell — no model
reload needed.


In [ ]:
# Cell 1: Imports + config
import sys, os, json
import numpy as np
import torch
import matplotlib.pyplot as plt

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from scripts.denoising_lab.denoising_lab import DenoisingLab, TrajectoryVisualizer

MODEL_PATH = "nvidia/GR00T-N1.6-3B"
EMBODIMENT_TAG = "robocasa_panda_omron"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Scene: GRPO scene seed, rebuilt in the robocasa venv by Cell 1.5 (no saved .npz).
# 103067 = slot 3 of this run's pool (100067 + j*1000).
ENV_NAME = "robocasa_panda_omron/CoffeeServeMug_PandaOmron_Env"
SEED = 103067
SIM_PYTHON = os.path.join(REPO_ROOT, "gr00t/eval/sim/robocasa/robocasa_uv/.venv/bin/python")
OBS_DIR = "/tmp/denoising_lab_seed_obs"   # capture cache; force=True in Cell 1.5 rebuilds
# Optional: the training log's "scene:" line for SEED ("layout=… style=… xml=… state=…").
# Cell 1.5 then raises if the rebuilt scene differs.
EXPECTED_SCENE = None

EEF_KEY = "end_effector_position"
N_SEEDS = 100              # seeds per checkpoint (matches v1 Cell 11)

# LoRA config — MUST match what the checkpoints were trained with.
# (GRPOConfig defaults; this run's TB config table confirms rank 16 / alpha 32.)
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.0

# Also sweep the BASE model (no LoRA) by disabling the injected adapter. Produces the
# same composite figure and an entry in the cross-checkpoint summary. See Cell 4.
INCLUDE_BASE_MODEL = True

# Every figure's data is also saved to PLOT_DATA_DIR/<name>.npz so the plots can be
# re-drawn (e.g. smoothed) offline without the GPU. File list: "Saved plot data" below.
PLOT_DATA_DIR = os.path.join(REPO_ROOT, "scripts", "denoising_lab", "notebooks", "plot_data_v5")
os.makedirs(PLOT_DATA_DIR, exist_ok=True)


def save_plot_data(name, meta, **arrays):
    """Write arrays + a JSON meta string to PLOT_DATA_DIR/<name>.npz (loads without pickle)."""
    path = os.path.join(PLOT_DATA_DIR, f"{name}.npz")
    np.savez(path, meta_json=json.dumps(meta), **{k: np.asarray(v) for k, v in arrays.items()})
    print(f"Plot data saved: {path}")

# Checkpoints to compare: {short label: dir relative to REPO_ROOT holding lora_weights.pt}.
# The short label names the figures and summary rows; the full run name overflows them.
CHECKPOINTS = {
    "res7_velanch_1.0_i3/iter_0012": "grpo_data/full_pool4_v5/lr5e-5PAWS_plam0.125_nlam0_loclip0.005_hiclip0.2_kll0.1_klb0.1_tratio2.25_mbrenorm_BMPARD_nojitterpair_3epoch_gs12_ng4_mb128_IAG0.15_FFP0_pool4_loclipmse0.8_failfilter_res3_velanch0.45_velanch_i3_maskloss_fixedstd_iternorm_res7_velanch_1.0_i3/iter_0012/",
}

n_runs = len(CHECKPOINTS) + (1 if INCLUDE_BASE_MODEL else 0)
print(f"Device: {DEVICE}")
print(f"{len(CHECKPOINTS)} checkpoints"
      f"{' + base model' if INCLUDE_BASE_MODEL else ''} = {n_runs} runs, {N_SEEDS} seeds each")
print(f"Plot data -> {PLOT_DATA_DIR}")

### Saved plot data

Every figure's data is written to `PLOT_DATA_DIR` (`scripts/denoising_lab/notebooks/plot_data_v5/`),
one `.npz` per figure family and observation, so the plots can be re-drawn (e.g. smoothed)
offline without the GPU. `<obs>` is the observation file stem, e.g.
`CoffeeServeMug_seed103067_step000`.

| file | figures |
|---|---|
| `camera_views__<obs>.npz` | camera views (saved because the observation itself is only a `/tmp` cache) |
| `seed_spread__<obs>.npz` | composite seed-spread figure per run (3D panel, X/Y/Z traces, mean±std) and the final-std summary bar chart |

`seed_spread` files have the same layout as v4's, so the same re-plot code reads both. The
jerk stats are not stored separately: they follow from `all_pos` with the formulas in
`run_seed_sweep`. Each file holds a `meta_json` string with the observation path, task,
seed + scene fingerprint (`step_info`), checkpoint paths and array shapes.

```python
d = np.load(os.path.join(PLOT_DATA_DIR, "seed_spread__CoffeeServeMug_seed103067_step000.npz"))
meta = json.loads(d["meta_json"].item())
d["all_pos"].shape   # (run, seed, timestep, xyz)
```


In [ ]:
# Cell 1.5: Rebuild SEED's scene in the robocasa venv and save its first observation.
# Runs capture_seed_obs.py there as a subprocess (robocasa isn't importable in this
# venv). Cached in OBS_DIR; force=True rebuilds.
import json
import subprocess

CAPTURE_SCRIPT = os.path.join(REPO_ROOT, "scripts/denoising_lab/eval/capture_seed_obs.py")


def capture_seed_observation(seed, env_name=ENV_NAME, force=False, expected_scene=None):
    task = env_name.split("/")[-1].removesuffix("_PandaOmron_Env")
    out = os.path.join(OBS_DIR, f"{task}_seed{seed}_step000.npz")
    if force or not os.path.exists(out):
        if not os.path.exists(SIM_PYTHON):
            raise FileNotFoundError(f"robocasa venv not found: {SIM_PYTHON} "
                                    "(see gr00t/eval/sim/robocasa/setup_RoboCasa.sh)")
        cmd = [SIM_PYTHON, "-u", CAPTURE_SCRIPT, "--env-name", env_name,
               "--seed", str(seed), "--out", out]
        proc = subprocess.Popen(cmd, cwd=REPO_ROOT, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT,
                                env={**os.environ, "GRPO_CLEAN_OUTPUT": "1"})
        for line in proc.stdout:
            print("  [sim]", line, end="")
        if proc.wait() != 0:
            raise RuntimeError(f"capture_seed_obs.py exited {proc.returncode}; "
                               "see the [sim] lines above")
    with np.load(out, allow_pickle=True) as z:
        scene = json.loads(str(z["__step_info__"]))["scene_fingerprint"]
    print(f"seed {seed}: {out}\n  scene: {scene}")
    if expected_scene is not None and scene != expected_scene:
        raise RuntimeError(f"seed {seed} rebuilt a different scene than training:\n"
                           f"  got      {scene}\n  expected {expected_scene}")
    return out


OBS_PATH = capture_seed_observation(SEED, expected_scene=EXPECTED_SCENE)


In [ ]:
# Cell 2: Load model (once)
lab = DenoisingLab(MODEL_PATH, EMBODIMENT_TAG, device=DEVICE)
print(f"Model loaded. Action horizon: {lab.action_horizon}, Action dim: {lab.action_dim}")
print(f"Inference timesteps: {lab.num_inference_timesteps}, Timestep buckets: {lab.num_timestep_buckets}")

In [ ]:
# Cell 2.5: Inject LoRA ONCE into the DiT — do NOT load a checkpoint yet, do NOT merge.
#
# We inject the adapter shell here and reuse it for every checkpoint: the loop below
# calls load_lora_checkpoint(...) to swap each checkpoint's A/B matrices into these
# same modules. We deliberately skip merge_lora_weights() — merging is irreversible
# and would bake one checkpoint into the base weights, making it impossible to cleanly
# load the next one OR to disable the adapter for the base-model run. The cost is a
# small per-forward LoRA matmul (a few extra seconds per 100-seed sweep), which is fine.
import sys as _sys
_grpo_dir = os.path.join(REPO_ROOT, "scripts", "grpo")
if _grpo_dir not in _sys.path:
    _sys.path.insert(0, _grpo_dir)

# disabled_adapters: context manager that zeros the LoRA contribution for the base-model
# sweep — the DiT then produces its pretrained velocity field, no second copy in VRAM.
from lora_dit import (
    apply_lora_to_dit,
    load_lora_checkpoint,
    disabled_adapters,
    print_trainable_params,
)

apply_lora_to_dit(lab.model, rank=LORA_RANK, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)

# Pin the freshly-injected LoRA Linears to the DiT's device/dtype. Done ONCE:
# load_lora_checkpoint copies weights in place via load_state_dict, so loaded adapters
# inherit this device/dtype on every subsequent load (no re-.to() needed).
lab.model.action_head.model.to(device=lab.device, dtype=lab.dtype)
lab.model.eval()

print_trainable_params(lab.model)  # ~14.5M LoRA params, rest frozen

## Per-checkpoint seed sweep

`run_seed_sweep` reproduces v1 Cell 11 ("Compare seeds") for a single
`(features, checkpoint)` pair, but draws **one composite figure** per adapter:

```
+------------+  +----+ +----+ +----+   <- X/Y/Z component traces (100 seeds)
|            |  +----+ +----+ +----+
|  3D EEF    |
|  traj      |  +----+ +----+ +----+   <- x/y/z mean ± 1 std spread
|            |  +----+ +----+ +----+
+------------+
```

`analyze_all_checkpoints` drives it across the base model (adapters disabled) and
every checkpoint for one observation, encoding backbone features just once.

### Jerk / smoothness stats

Printed under the spread stats in the same `mean / var / std / range` layout. Each
line is taken **across seeds** of one number per seed. The EEF path is
`pos = [0, cumsum(deltas)]`, so jerk = 3rd difference of `pos` = 2nd difference of the
decoded per-step deltas: 14 windows over the 16-step chunk.

- **per-seed mean over all timesteps**: mean `abs(jerk)` over the 14 windows.
- **final timestep only**: `abs(jerk)` at the last window (path steps 13–16, never
  executed at `n_action_steps=8`).
- **normalized jerk**: `sum abs(jerk) / path length`, scale-free. The `3D` row,
  `sum ‖jerk‖ / path length`, is v4's `pathjerk`. For scale only: on a different observation (seed 305067,
  step 10) v4 read 0.069 for the base model and 0.52 for the overfit `iter_0011`.

Raw jerk grows with path length, so a policy that moves further reads rougher even
if its paths are just as smooth. The normalized block separates "rougher" from
"bigger".


In [ ]:
# Cell 3: run_seed_sweep — reproduces v1 Cell 11 ("Compare seeds") for one
# (features, checkpoint) pair. Sweeps n_seeds noise seeds through the 4-step denoiser,
# decodes each to an EEF trajectory, prints per-component spread stats and jerk /
# smoothness stats, then draws ONE composite figure: a large 3D EEF-trajectory panel
# (left), the X/Y/Z component traces (top-right), and the x/y/z mean+-std spread
# (bottom-right). Returns a stats dict for the cross-checkpoint summary.
def _seed_stats_line(v, l):
    """Print mean/var/std/range across seeds of a per-seed quantity v, shape (n_seeds,)."""
    print(f"  {l}: mean={v.mean():.4g}, \tvar={v.var():.4g}, \tstd={v.std():.4g}, \trange={np.ptp(v):.4g}")


def run_seed_sweep(lab, features, label, n_seeds=N_SEEDS, eef_key=EEF_KEY, show_plots=True):
    viz_seeds = TrajectoryVisualizer()
    for seed in range(n_seeds):
        r = lab.denoise(features, seed=seed)
        d = lab.decode_raw_actions(r.action_pred, features.states)
        viz_seeds.add_trajectory(d, f"seed={seed}", eef_key=eef_key)

    # Stack all trajectories: shape (n_seeds, T+1, 3)
    all_pos = np.stack([t["positions"] for t in viz_seeds.trajectories])
    mean = all_pos.mean(axis=0)   # (T+1, 3)
    var = all_pos.var(axis=0)     # (T+1, 3)
    rng = all_pos.max(axis=0) - all_pos.min(axis=0)  # (T+1, 3)

    comp = ["x", "y", "z"]
    print("Across all timesteps:")
    for i, l in enumerate(comp):
        print(f"  {l}: mean={mean[:, i].mean():.4f}, \tvar={var[:, i].mean():.4f}, \tstd={np.sqrt(var[:, i]).mean():.4f}, \trange={rng[:, i].mean():.4f}")
    print("Across final timestep only:")
    for i, l in enumerate(comp):
        print(f"  {l}: mean={mean[-1, i]:.4f}, \tvar={var[-1, i]:.4f}, \tstd={np.sqrt(var[-1, i]):.4f}, \trange={rng[-1, i]:.4f}")

    # Jerk = 3rd difference of the EEF path = 2nd difference of the per-step deltas.
    # Every line is across seeds of one number per seed (see the markdown above).
    jerk = np.diff(all_pos, n=3, axis=1)                                     # (n_seeds, T-2, 3)
    jabs = np.abs(jerk)
    path_len = np.linalg.norm(np.diff(all_pos, axis=1), axis=2).sum(axis=1)  # (n_seeds,)
    denom = np.maximum(path_len, 1e-9)
    norm_jerk = jabs.sum(axis=1) / denom[:, None]                            # (n_seeds, 3)
    pathjerk = np.linalg.norm(jerk, axis=2).sum(axis=1) / denom              # (n_seeds,)
    print("Jerk |3rd diff of EEF path|, per-seed mean over all timesteps:")
    for i, l in enumerate(comp):
        _seed_stats_line(jabs[:, :, i].mean(axis=1), l)
    print("Jerk |3rd diff of EEF path|, final timestep only:")
    for i, l in enumerate(comp):
        _seed_stats_line(jabs[:, -1, i], l)
    print("Normalized jerk = sum|jerk| / path length, per seed (scale-free; 3D = v4 pathjerk):")
    for i, l in enumerate(comp):
        _seed_stats_line(norm_jerk[:, i], l)
    _seed_stats_line(pathjerk, "3D")

    if show_plots:
        # ONE composite figure per adapter: 3D (left, full height) + 2x3 grid (right).
        comp_up = ["X", "Y", "Z"]
        fig = plt.figure(figsize=(20, 8), constrained_layout=True)
        gs = fig.add_gridspec(2, 5)

        # Left: 3D EEF trajectory — one line per seed, shared start (^) and mean end (square).
        ax3d = fig.add_subplot(gs[:, 0:2], projection="3d")
        for s in range(all_pos.shape[0]):
            p = all_pos[s]
            ax3d.plot(p[:, 0], p[:, 1], p[:, 2], "-", color="C0", linewidth=0.6, alpha=0.4)
        ax3d.scatter(*all_pos[0, 0], marker="^", s=60, color="k", zorder=5)
        ax3d.scatter(*mean[-1], marker="s", s=60, color="r", zorder=5)
        # Equal aspect via bounding box (mirrors TrajectoryVisualizer.plot_eef_3d).
        flat = all_pos.reshape(-1, 3)
        bmin, bmax = flat.min(axis=0), flat.max(axis=0)
        mid = (bmin + bmax) / 2
        half = (bmax - bmin).max() / 2
        if half < 1e-8:
            half = 0.01
        ax3d.set_xlim(mid[0] - half, mid[0] + half)
        ax3d.set_ylim(mid[1] - half, mid[1] + half)
        ax3d.set_zlim(mid[2] - half, mid[2] + half)
        ax3d.set_xlabel("X"); ax3d.set_ylabel("Y"); ax3d.set_zlabel("Z")
        ax3d.set_title(f"3D EEF trajectory ({n_seeds} seeds)", fontsize=10)

        # Top-right: raw X/Y/Z component traces (one line per seed).
        for i in range(3):
            axc = fig.add_subplot(gs[0, 2 + i])
            axc.plot(all_pos[:, :, i].T, color="C0", linewidth=0.5, alpha=0.25)
            axc.set_title(f"EEF {comp_up[i]} ({n_seeds} seeds)", fontsize=9)
            axc.set_xlabel("timestep", fontsize=8)
            axc.grid(True, alpha=0.3)

        # Bottom-right: x/y/z mean +- 1 std spread.
        for i in range(3):
            axs = fig.add_subplot(gs[1, 2 + i])
            std_i = np.sqrt(var[:, i])
            axs.plot(mean[:, i], color="C1", label="mean")
            axs.fill_between(range(mean.shape[0]), mean[:, i] - std_i, mean[:, i] + std_i,
                             alpha=0.3, color="C1", label="±1 std")
            axs.set_title(f"{comp[i]} mean±std", fontsize=9)
            axs.set_xlabel("timestep", fontsize=8)
            axs.grid(True, alpha=0.3)
            if i == 0:
                axs.legend(fontsize=7)

        fig.suptitle(f"Seed spread — {label}", fontsize=13)
        plt.show()

    return {
        "label": label,
        "all_pos": all_pos,  # (n_seeds, T+1, 3) — every plotted seed line
        "mean": mean,
        "var": var,
        "range": rng,
        "final_std": np.sqrt(var[-1]),           # (3,) std at final timestep
        "mean_std": np.sqrt(var).mean(axis=0),   # (3,) std averaged over timesteps
        "jerk_mean": jabs.mean(axis=(0, 1)),     # (3,) mean |jerk| per step
        "jerk_final": jabs[:, -1].mean(axis=0),  # (3,) mean |jerk| at the last window
        "norm_jerk": norm_jerk.mean(axis=0),     # (3,) sum|jerk| / path length
        "pathjerk": float(pathjerk.mean()),      # sum||jerk|| / path length (v4 pathjerk)
        "path_len": float(path_len.mean()),
    }

In [ ]:
# Cell 4: analyze_all_checkpoints — encode one observation, then run the seed sweep
# for the base model (adapters disabled) and every checkpoint in CHECKPOINTS. Features
# are encoded ONCE (the backbone is LoRA-free); each checkpoint's adapter is loaded into
# the already-injected DiT, fully overwriting the previous checkpoint's A/B matrices.
def analyze_all_checkpoints(obs_path, n_seeds=N_SEEDS, show_obs=True, include_base=INCLUDE_BASE_MODEL):
    obs = DenoisingLab.load_observation(obs_path)
    obs_tag = os.path.splitext(os.path.basename(obs_path))[0]
    task = list(obs["annotation.human.action.task_description"])
    with np.load(obs_path, allow_pickle=True) as z:  # seed + scene fingerprint, for the plot-data meta
        step_info = json.loads(str(z["__step_info__"])) if "__step_info__" in z.files else None
    print("Observation:", obs_path)
    print("Task:", obs["annotation.human.action.task_description"])
    if show_obs:
        fig = DenoisingLab.plot_camera_views(obs, figsize=(14, 4))
        plt.suptitle(f"Camera views — {os.path.basename(obs_path)}", fontsize=12)
        # OBS_DIR is a /tmp cache, so save the drawn images themselves (taken from the figure).
        cams = [(ax.get_title(), np.asarray(ax.get_images()[0].get_array())) for ax in fig.axes]
        save_plot_data(
            f"camera_views__{obs_tag}",
            meta={"obs_path": obs_path, "task": task, "step_info": step_info,
                  "arrays": {"names": "(camera,) panel titles, left to right",
                             "image_<i>": "(H, W, 3) uint8, panel i"}},
            names=[n for n, _ in cams], **{f"image_{i}": img for i, (_, img) in enumerate(cams)},
        )
        plt.show()

    features = lab.encode_features_from_sim_obs(obs)  # EXPENSIVE — once per obs
    print(f"Backbone features: {features.backbone_features.shape}")

    results, ckpt_of = {}, {}

    # Base model (no LoRA): disable the injected adapters so the DiT produces its
    # pretrained velocity field. Independent of whichever checkpoint is currently
    # loaded, so we run it FIRST — before touching any adapter weights.
    if include_base:
        label = "BASE (no LoRA)"
        print("\n" + "=" * 90)
        print(f"CHECKPOINT: {label}")
        print("=" * 90)
        with disabled_adapters(lab.model.action_head.model):
            results[label] = run_seed_sweep(lab, features, label, n_seeds=n_seeds)
        ckpt_of[label] = ""  # no LoRA

    for label, ckpt_rel in CHECKPOINTS.items():
        ckpt = os.path.join(REPO_ROOT, ckpt_rel)
        print("\n" + "=" * 90)
        print(f"CHECKPOINT: {label}")
        print("=" * 90)
        load_lora_checkpoint(lab.model, ckpt)  # overwrites the previous adapter in place
        results[label] = run_seed_sweep(lab, features, label, n_seeds=n_seeds)
        ckpt_of[label] = ckpt

    # Data behind the composite figures above and the summary bar chart (Cell 6).
    runs = list(results)
    save_plot_data(
        f"seed_spread__{obs_tag}",
        meta={"obs_path": obs_path, "task": task, "step_info": step_info,
              "ckpt_paths": [ckpt_of[k] for k in runs], "n_seeds": n_seeds, "eef_key": EEF_KEY,
              "arrays": {"all_pos": "(run, seed, timestep, xyz) EEF path from 0; 3D panel + X/Y/Z traces",
                         "mean/std": "(run, timestep, xyz) across seeds; mean±std panels",
                         "final_std": "(run, xyz) std at the final timestep; summary bar chart"}},
        runs=runs,
        timestep=np.arange(results[runs[0]]["all_pos"].shape[1]),
        all_pos=np.stack([results[k]["all_pos"] for k in runs]),
        mean=np.stack([results[k]["mean"] for k in runs]),
        std=np.sqrt(np.stack([results[k]["var"] for k in runs])),
        final_std=np.stack([results[k]["final_std"] for k in runs]),
    )
    return features, results

In [ ]:
# Cell 5: Run the full comparison (base model + all checkpoints) for OBS_PATH, the
# first observation of scene seed SEED (Cell 1.5). For another pool scene:
# analyze_all_checkpoints(capture_seed_observation(100067)).
_features, results = analyze_all_checkpoints(OBS_PATH)

## Cross-checkpoint summary

Final-timestep EEF-position std across the seeds, per checkpoint (and the base model).
Higher std = more action diversity driven by the initial noise; lower std = the
checkpoint has become more "decisive" / peaked for this observation.

The second table lists the jerk / smoothness means per run: `jerk_x/y/z` = mean
`abs(jerk)` per step, `pathjerk` = `sum ‖jerk‖ / path length` (scale-free),
`path_len` = mean EEF path length.


In [ ]:
# Cell 6: Cross-checkpoint summary — final-timestep EEF position std across seeds,
# plus the jerk / smoothness means.
labels_short = list(results.keys())
final_std = np.array([results[k]["final_std"] for k in labels_short])  # (n_ckpt, 3)

print(f"{'checkpoint':52s} {'std_x':>8s} {'std_y':>8s} {'std_z':>8s} {'mean':>9s}")
for k, fs in zip(labels_short, final_std):
    print(f"{k:52s} {fs[0]:8.4f} {fs[1]:8.4f} {fs[2]:8.4f} {fs.mean():9.4f}")

# Jerk / smoothness means per run (definitions in the seed-sweep markdown).
print(f"\n{'checkpoint':52s} {'jerk_x':>8s} {'jerk_y':>8s} {'jerk_z':>8s} {'pathjerk':>9s} {'path_len':>9s}")
for k in labels_short:
    jm = results[k]["jerk_mean"]
    print(f"{k:52s} {jm[0]:8.5f} {jm[1]:8.5f} {jm[2]:8.5f} "
          f"{results[k]['pathjerk']:9.5f} {results[k]['path_len']:9.3f}")

x = np.arange(len(labels_short))
w = 0.25
fig, ax = plt.subplots(figsize=(15, 6))
for i, l in enumerate(["x", "y", "z"]):
    ax.bar(x + (i - 1) * w, final_std[:, i], w, label=f"std {l}")
ax.set_xticks(x)
ax.set_xticklabels(labels_short, rotation=70, ha="right", fontsize=7)
ax.set_ylabel("final-timestep EEF std across seeds")
ax.set_title(f"Action diversity by checkpoint — {os.path.basename(OBS_PATH)}")
ax.legend()
plt.tight_layout()
plt.show()